In [1]:
# Installing required packages
!pip install -q pyspark findspark
!pip install pyarrow
!pip install pandas
!pip install numpy

In [2]:
import pandas as pd
from pyspark import SparkContext, SparkConf
from pyspark.ml.feature import VectorAssembler, StringIndexer, MinMaxScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import SparkSession
import findspark
findspark.init()
# Creating a Spark session.
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("OptimizingUrbanMobilitySolutions").getOrCreate()

In [3]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [4]:
trip_df = spark.read.csv("/content/taxi_trip_data.csv", header=True, inferSchema=True)
zone_df = spark.read.csv("/content/taxi_zone_geo.csv", header=True, inferSchema=True)

In [5]:
trip_df.show()
zone_df.show()

+---------+----------------+----------------+---------------+-------------+---------+------------------+------------+-----------+-----+-------+----------+------------+-------------+------------------+-------------------+
|vendor_id| pickup_datetime|dropoff_datetime|passenger_count|trip_distance|rate_code|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|imp_surcharge|pickup_location_id|dropoff_location_id|
+---------+----------------+----------------+---------------+-------------+---------+------------------+------------+-----------+-----+-------+----------+------------+-------------+------------------+-------------------+
|        1| 5/11/2018 17:40| 5/11/2018 17:55|              1|          1.6|        1|                 N|           1|       11.5|  1.0|    0.5|       0.0|         0.0|          0.3|                48|                 68|
|        2| 3/22/2018 23:01| 3/22/2018 23:25|              1|         9.52|        1|                 N|           1

In [6]:
# === 1b: Handle missing values (explicit and implicit) ===

# Step 1: Replace empty strings and string placeholders with nulls (excluding tip_amount)
for column_name in trip_df.columns:
    if column_name != "tip_amount":
        trip_df = trip_df.withColumn(column_name,
            when((col(column_name) == "") |
                 (col(column_name).isin("0", "0.0", "null", "NULL")), None)
            .otherwise(col(column_name))
        )

# Step 2: Cast numeric fields to proper types
numeric_casts = {
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "trip_distance": "double",
    "passenger_count": "int"
}

for col_name, target_type in numeric_casts.items():
    trip_df = trip_df.withColumn(col_name, col(col_name).cast(target_type))

# Step 3: Replace missing or zero passenger_count with 1
trip_df = trip_df.withColumn("passenger_count",
    when((col("passenger_count").isNull()) | (col("passenger_count") == 0), 1)
    .otherwise(col("passenger_count"))
)

# Step 4: Drop rows with nulls in essential columns (excluding tip_amount)
essential_columns = [
    "pickup_datetime", "dropoff_datetime", "fare_amount",
    "extra", "mta_tax", "tolls_amount",
    "trip_distance", "passenger_count",
    "pickup_location_id", "dropoff_location_id"
]

trip_df = trip_df.dropna(subset=essential_columns)

In [7]:
trip_df.show()

+---------+----------------+----------------+---------------+-------------+---------+------------------+------------+-----------+-----+-------+----------+------------+-------------+------------------+-------------------+
|vendor_id| pickup_datetime|dropoff_datetime|passenger_count|trip_distance|rate_code|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|imp_surcharge|pickup_location_id|dropoff_location_id|
+---------+----------------+----------------+---------------+-------------+---------+------------------+------------+-----------+-----+-------+----------+------------+-------------+------------------+-------------------+
|        2| 12/6/2018 16:38| 12/6/2018 17:29|              1|         8.91|        1|                 N|           1|       36.0|  1.0|    0.5|      8.71|        5.76|          0.3|               138|                 90|
|        2| 3/27/2018 17:05| 3/27/2018 18:01|              2|        16.99|        2|                 N|           2

In [8]:
# === 1c: Drop duplicate rows and irrelevant columns ===

# Drop duplicate rows
trip_df = trip_df.dropDuplicates()

# Drop irrelevant columns based on project analysis
columns_to_drop = ["vendor_id", "rate_code", "store_and_fwd_flag", "imp_surcharge", "total_amount"]
trip_df = trip_df.drop(*columns_to_drop)

# Print final schema after cleaning
print("Cleaned Trip Dataset Schema:")
trip_df.printSchema()

# Show sample of cleaned data
trip_df.show(10, truncate=False)


Cleaned Trip Dataset Schema:
root
 |-- pickup_datetime: string (nullable = true)
 |-- dropoff_datetime: string (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)

+----------------+----------------+---------------+-------------+------------+-----------+-----+-------+----------+------------+------------------+-------------------+
|pickup_datetime |dropoff_datetime|passenger_count|trip_distance|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|pickup_location_id|dropoff_location_id|
+----------------+----------------+---------------+-------------+------------+----------

In [9]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# Define the correct datetime format
datetime_format = "MM/dd/yyyy HH:mm"

# Parse pickup and dropoff datetime strings
trip_df = trip_df.withColumn("pickup_datetime", to_timestamp("pickup_datetime", datetime_format))
trip_df = trip_df.withColumn("dropoff_datetime", to_timestamp("dropoff_datetime", datetime_format))

# Compute trip duration in minutes
trip_df = trip_df.withColumn(
    "trip_duration_minutes",
    (unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60
)

# Filter out rows with null or non-positive durations
trip_df = trip_df.filter(col("trip_duration_minutes").isNotNull() & (col("trip_duration_minutes") > 0))
trip_df = trip_df.filter(col("passenger_count").isNotNull() & (col("passenger_count") > 0))
trip_df = trip_df.filter(col("trip_distance").isNotNull() & (col("trip_distance") > 0))

In [10]:
# Compute total trip cost

trip_df = trip_df.withColumn(
    "total_trip_cost",
    col("fare_amount") + col("extra") + col("mta_tax") + col("tip_amount") + col("tolls_amount")
)

In [11]:
trip_df.show()

+-------------------+-------------------+---------------+-------------+------------+-----------+-----+-------+----------+------------+------------------+-------------------+---------------------+------------------+
|    pickup_datetime|   dropoff_datetime|passenger_count|trip_distance|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|pickup_location_id|dropoff_location_id|trip_duration_minutes|   total_trip_cost|
+-------------------+-------------------+---------------+-------------+------------+-----------+-----+-------+----------+------------+------------------+-------------------+---------------------+------------------+
|2018-02-22 18:58:00|2018-02-22 19:23:00|              1|        11.88|           1|       33.5|  1.0|    0.5|      8.21|        5.76|               138|                127|                 25.0|             48.97|
|2018-03-22 19:17:00|2018-03-22 20:04:00|              1|         20.6|           1|       52.0|  4.5|    0.5|      12.6|        5.76|      

In [12]:
# === 1e: Enrich dataset with zone-based location info from taxizonegeo.csv ===

# Rename zone columns for pickup
zone_df_pickup = zone_df.withColumnRenamed("zone_id", "pickup_location_id") \
                        .withColumnRenamed("Borough", "pickup_borough") \
                        .withColumnRenamed("zone_name", "pickup_zone") \
                        .withColumnRenamed("zone_geom", "pickup_service_zone")

# Join pickup zone data
trip_df = trip_df.join(zone_df_pickup, on="pickup_location_id", how="left")

# Rename zone columns for dropoff
zone_df_dropoff = zone_df.withColumnRenamed("zone_id", "dropoff_location_id") \
                         .withColumnRenamed("Borough", "dropoff_borough") \
                         .withColumnRenamed("zone_name", "dropoff_zone") \
                         .withColumnRenamed("zone_geom", "dropoff_service_zone")

# Join dropoff zone data
trip_df = trip_df.join(zone_df_dropoff, on="dropoff_location_id", how="left")


In [13]:
trip_df.show()

+-------------------+------------------+-------------------+-------------------+---------------+-------------+------------+-----------+-----+-------+----------+------------+---------------------+------------------+--------------------+--------------+--------------------+--------------------+---------------+--------------------+
|dropoff_location_id|pickup_location_id|    pickup_datetime|   dropoff_datetime|passenger_count|trip_distance|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|trip_duration_minutes|   total_trip_cost|         pickup_zone|pickup_borough| pickup_service_zone|        dropoff_zone|dropoff_borough|dropoff_service_zone|
+-------------------+------------------+-------------------+-------------------+---------------+-------------+------------+-----------+-----+-------+----------+------------+---------------------+------------------+--------------------+--------------+--------------------+--------------------+---------------+--------------------+
|         

In [14]:
# Cleaning after the join
# drop pickup zone and dropoff zone columns
trip_df = trip_df.drop("pickup_service_zone", "dropoff_service_zone")

# drop null values after the join in all columns
trip_df = trip_df.dropna()

trip_df.show()

+-------------------+------------------+-------------------+-------------------+---------------+-------------+------------+-----------+-----+-------+----------+------------+---------------------+------------------+--------------------+--------------+--------------------+---------------+
|dropoff_location_id|pickup_location_id|    pickup_datetime|   dropoff_datetime|passenger_count|trip_distance|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|trip_duration_minutes|   total_trip_cost|         pickup_zone|pickup_borough|        dropoff_zone|dropoff_borough|
+-------------------+------------------+-------------------+-------------------+---------------+-------------+------------+-----------+-----+-------+----------+------------+---------------------+------------------+--------------------+--------------+--------------------+---------------+
|                127|               138|2018-02-22 18:58:00|2018-02-22 19:23:00|              1|        11.88|           1|       33.5| 

In [15]:
spark

In [16]:
trip_df.createOrReplaceTempView("trips")
zone_df.createOrReplaceTempView("zones")


In [17]:
spark.sql("""
WITH time_payment AS (
  SELECT *,
         CASE
           WHEN HOUR(TO_TIMESTAMP(pickup_datetime, 'M/d/yyyy H:mm')) BETWEEN 5 AND 11 THEN 'Morning'
           WHEN HOUR(TO_TIMESTAMP(pickup_datetime, 'M/d/yyyy H:mm')) BETWEEN 12 AND 17 THEN 'Afternoon'
           ELSE 'Evening'
         END AS time_of_day
  FROM trips
),
ranked AS (
  SELECT time_of_day, payment_type, COUNT(*) AS trip_count,
         DENSE_RANK() OVER (PARTITION BY time_of_day ORDER BY COUNT(*) DESC) AS rnk
  FROM time_payment
  GROUP BY time_of_day, payment_type
)
SELECT time_of_day, payment_type, trip_count
FROM ranked
WHERE rnk = 1
ORDER BY time_of_day
""").show()


+-----------+------------+----------+
|time_of_day|payment_type|trip_count|
+-----------+------------+----------+
|  Afternoon|           1|      4199|
|    Evening|           1|     10207|
|    Morning|           1|       461|
+-----------+------------+----------+



In [18]:
spark.sql("""SELECT passenger_count, ROUND(AVG(tip_amount), 2) AS avg_tip
FROM trips
GROUP BY passenger_count
ORDER BY passenger_count
""").show()

+---------------+-------+
|passenger_count|avg_tip|
+---------------+-------+
|              1|   6.22|
|              2|   5.88|
|              3|   5.84|
|              4|   5.18|
|              5|    6.2|
|              6|   6.38|
+---------------+-------+



In [19]:
spark.sql("""
SELECT time_of_day, zone.zone_name AS pickup_zone, COUNT(*) AS trip_count
FROM (
  SELECT *,
         CASE
           WHEN HOUR(TO_TIMESTAMP(pickup_datetime, 'M/d/yyyy H:mm')) BETWEEN 5 AND 11 THEN 'Morning'
           WHEN HOUR(TO_TIMESTAMP(pickup_datetime, 'M/d/yyyy H:mm')) BETWEEN 12 AND 17 THEN 'Afternoon'
           ELSE 'Evening'
         END AS time_of_day
  FROM trips
) t
JOIN zones zone ON t.pickup_location_id = zone.zone_id
GROUP BY time_of_day, zone.zone_name
ORDER BY trip_count DESC
LIMIT 5
""").show()


+-----------+-----------------+----------+
|time_of_day|      pickup_zone|trip_count|
+-----------+-----------------+----------+
|    Evening|LaGuardia Airport|      6170|
|  Afternoon|LaGuardia Airport|      1914|
|  Afternoon|      JFK Airport|      1213|
|    Evening|      JFK Airport|      1200|
|    Evening|      Murray Hill|       303|
+-----------+-----------------+----------+



In [20]:
spark.sql("""
SELECT pickup_borough, SUM(total_trip_cost) AS total_revenue, COUNT(*) AS trip_volume
FROM trips
WHERE pickup_borough IS NOT NULL
GROUP BY pickup_borough
ORDER BY total_revenue DESC
""").show()

+--------------+------------------+-----------+
|pickup_borough|     total_revenue|trip_volume|
+--------------+------------------+-----------+
|        Queens| 554747.1100000234|      10926|
|     Manhattan| 348197.4600000139|       7340|
|      Brooklyn| 4661.660000000006|         95|
|         Bronx| 2876.590000000002|         55|
| Staten Island|59.019999999999996|          1|
+--------------+------------------+-----------+



In [21]:
spark.sql("""
SELECT
    t.trip_duration_minutes,
    t.fare_amount,
    pz.zone_name AS pickup_zone,
    dz.zone_name AS dropoff_zone,
    t.payment_type
FROM
    trips t
JOIN
    zones pz ON t.pickup_location_id = pz.zone_id
JOIN
    zones dz ON t.dropoff_location_id = dz.zone_id
ORDER BY
    t.trip_duration_minutes DESC
LIMIT 5
""").show(truncate=False)

+---------------------+-----------+----------------------------+------------------------------+------------+
|trip_duration_minutes|fare_amount|pickup_zone                 |dropoff_zone                  |payment_type|
+---------------------+-----------+----------------------------+------------------------------+------------+
|1440.0               |65.0       |JFK Airport                 |Van Cortlandt Village         |1           |
|1439.0               |40.5       |Midtown North               |Flatbush/Ditmas Park          |1           |
|1436.0               |16.5       |Penn Station/Madison Sq West|Long Island City/Hunters Point|1           |
|1436.0               |41.0       |Lincoln Square East         |LaGuardia Airport             |1           |
|1436.0               |52.0       |Murray Hill                 |JFK Airport                   |1           |
+---------------------+-----------+----------------------------+------------------------------+------------+



In [22]:
spark.sql("""
SELECT
    pickup_borough,
    dropoff_borough,
    COUNT(*) AS trip_count,
    SUM(total_trip_cost) AS total_revenue
FROM
    trips
WHERE
    pickup_borough IS NOT NULL AND dropoff_borough IS NOT NULL
GROUP BY
    pickup_borough,
    dropoff_borough
ORDER BY
    trip_count DESC
LIMIT 10
""").show()

+--------------+---------------+----------+------------------+
|pickup_borough|dropoff_borough|trip_count|     total_revenue|
+--------------+---------------+----------+------------------+
|        Queens|      Manhattan|     10235| 514327.5800000188|
|     Manhattan|         Queens|      4700|232374.92000000319|
|     Manhattan|       Brooklyn|      1969| 82145.54999999962|
|        Queens|          Bronx|       525| 28964.30999999978|
|     Manhattan|          Bronx|       393| 17647.19999999997|
|     Manhattan|      Manhattan|       168|7301.8000000000075|
|        Queens|         Queens|       115| 7100.300000000006|
|     Manhattan|  Staten Island|       105| 8231.160000000002|
|      Brooklyn|      Manhattan|        48|1850.1399999999994|
|        Queens|  Staten Island|        31|2941.3499999999995|
+--------------+---------------+----------+------------------+



In [23]:
trip_df = trip_df.withColumn("high_tip", when(col("tip_amount") > 0.15 * col("fare_amount"), 1).otherwise(0))

In [24]:
trip_df = trip_df.withColumn("fare_per_mile", col("fare_amount") / col("trip_distance"))

In [25]:
trip_df = trip_df.withColumn("pickup_time", unix_timestamp("pickup_datetime"))

In [26]:
## Select features and normalize
features = ["passenger_count", "trip_distance", "trip_duration_minutes","pickup_time", "fare_amount", "fare_per_mile", "payment_type"]
assembler = VectorAssembler(inputCols=features, outputCol="unscaled_features")
assembled = assembler.transform(trip_df)

scaler = MinMaxScaler(inputCol="unscaled_features", outputCol="features")
scaler_model = scaler.fit(assembled)
scaled_data = scaler_model.transform(assembled).select("features", "high_tip")


In [27]:
train, test = scaled_data.randomSplit([0.7, 0.3], seed=42)

In [28]:
## Model 1: Logistic Regression
lr = LogisticRegression(featuresCol='features', labelCol='high_tip', maxIter=10)
lrModel = lr.fit(train)
lr_predictions = lrModel.transform(test)


#Logistic Regression gave strong results because the relationship between features like fare and trip duration closely followed a predictable trend.
#Its straightforward approach worked well with clean, engineered features and avoided overfitting.


In [29]:
## Model 2: Decision Tree
dt = DecisionTreeClassifier(featuresCol='features', labelCol='high_tip')
dtModel = dt.fit(train)
dt_predictions = dtModel.transform(test)

#The Decision Tree model learned specific rules from the data but performed slightly worse because it relied on a single set of decisions.
#It was more prone to errors when faced with unusual or inconsistent trips, limiting its overall accuracy.

In [30]:
## Model 3: Random Forest
rf = RandomForestClassifier(featuresCol='features', labelCol='high_tip')
rfModel = rf.fit(train)
rf_predictions = rfModel.transform(test)

#Random Forest performed the best by using many decision trees together to learn from different patterns in the data.
#By combining their outputs, it made more accurate and balanced predictions, reducing the impact of noise and outliers.

In [31]:
evaluator = BinaryClassificationEvaluator(labelCol="high_tip")
print("Logistic Regression Accuracy:", evaluator.evaluate(lr_predictions))
print("Decision Tree Accuracy:", evaluator.evaluate(dt_predictions))
print("Random Forest Accuracy:", evaluator.evaluate(rf_predictions))

Logistic Regression Accuracy: 0.8482844567336765
Decision Tree Accuracy: 0.8476178811964996
Random Forest Accuracy: 0.857887532751572
